1-Load

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('movement_features.csv')

print(f'Loaded {len(df)} rows')
print(f'Frames: {df["frame"].min()} to {df["frame"].max()}')
print(f'Players: {df["player_id"].nunique()}')
print(df.head())

2-Split in 1st and 2nd half

In [ ]:
# Fatigue = performance drop from first half to second half
# We split the video frames into two equal halves

total_frames = df['frame'].max()
midpoint     = total_frames // 2

print(f'Total frames: {total_frames}')
print(f'Midpoint: {midpoint}')
print(f'First half:  frames 0 to {midpoint}')
print(f'Second half: frames {midpoint+1} to {total_frames}')

first_half  = df[df['frame'] <= midpoint]
second_half = df[df['frame'] >  midpoint]

print(f'\nFirst half rows:  {len(first_half)}')
print(f'Second half rows: {len(second_half)}')

3-Find per player stats for each half

In [ ]:
def half_stats(half_df):
    return half_df.groupby('player_id').agg(
        avg_speed    = ('speed',     'mean'),
        avg_accel    = ('acceleration', lambda x: x[x > 0].mean()),  # only positive acceleration
        sprint_rate  = ('is_sprint', 'mean')   # fraction of frames that were sprints
    ).fillna(0)

first_stats  = half_stats(first_half)
second_stats = half_stats(second_half)

print('=== FIRST HALF STATS ===')
print(first_stats.round(2))
print('\n=== SECOND HALF STATS ===')
print(second_stats.round(2))

4-drop for each indicator

In [ ]:
# Combine both halves into one dataframe
stats = first_stats.join(second_stats, lsuffix='_first', rsuffix='_second')

# Calculate how much each metric dropped
# Positive drop = player got slower/less active = fatigued
# Negative drop = player got faster = not fatigued, clip to 0

stats['speed_drop']  = ((stats['avg_speed_first']   - stats['avg_speed_second'])   / stats['avg_speed_first'].replace(0, 1)).clip(0, 1)
stats['accel_drop']  = ((stats['avg_accel_first']   - stats['avg_accel_second'])   / stats['avg_accel_first'].replace(0, 1)).clip(0, 1)
stats['sprint_drop'] = ((stats['sprint_rate_first'] - stats['sprint_rate_second']) / stats['sprint_rate_first'].replace(0, 1)).clip(0, 1)

print('=== DROPS ===')
print(stats[['speed_drop', 'accel_drop', 'sprint_drop']].round(3))

5-Final fatigue scores

In [ ]:
# fatigue_score = 0.4 * speed_drop + 0.3 * accel_drop + 0.3 * sprint_drop
stats['fatigue_score'] = (
    0.4 * stats['speed_drop'] +
    0.3 * stats['accel_drop'] +
    0.3 * stats['sprint_drop']
).round(3)

# Add team info
team_info = df.groupby('player_id')['team_id'].first()
stats['team_id'] = team_info

# Sort by fatigue score
stats['fatigue_score'] = stats['fatigue_score'].fillna(0)

fatigue_df = stats[['team_id', 'fatigue_score', 'speed_drop', 'accel_drop', 'sprint_drop']].sort_values('fatigue_score', ascending=False)

print('=== FATIGUE SCORES ===')
print(fatigue_df.round(3).to_string())

6-Label

In [ ]:
# Give each player a human readable fatigue label
def fatigue_label(score):
    if score >= 0.7:
        return 'HIGH'
    elif score >= 0.4:
        return 'MEDIUM'
    else:
        return 'LOW'

fatigue_df['fatigue_level'] = fatigue_df['fatigue_score'].apply(fatigue_label)

print('=== FINAL FATIGUE REPORT ===')
print(fatigue_df[['team_id', 'fatigue_score', 'fatigue_level']].to_string())

print(f'\nTeam 0 avg fatigue: {fatigue_df[fatigue_df["team_id"]==0]["fatigue_score"].mean():.3f}')
print(f'Team 1 avg fatigue: {fatigue_df[fatigue_df["team_id"]==1]["fatigue_score"].mean():.3f}')

7-Save

In [ ]:
fatigue_df.to_csv('fatigue_scores.csv')

print('Saved to fatigue_scores.csv')
print(f'\nMost fatigued player:  #{fatigue_df["fatigue_score"].idxmax()} — score {fatigue_df["fatigue_score"].max():.3f}')
print(f'Freshest player:       #{fatigue_df["fatigue_score"].idxmin()} — score {fatigue_df["fatigue_score"].min():.3f}')